[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# YAML y sintaxis de workflows.

Este cuaderno desarrolla YAML, triggers, jobs, steps, expresiones y variables con el formato editorial de Pythonista.

## Objetivos.

- Entender la estructura minima de un workflow.
- Identificar los triggers principales.
- Reconocer el papel de expresiones y contextos.
- Encadenar jobs con `needs:` y condicionarlos con `if:`.
- Persistir resultados entre jobs con artefactos.
- Controlar timeouts y fallos esperados.

## Estructura minima de un workflow.

Un workflow se define dentro de `.github/workflows/` y normalmente contiene bloques como `name`, `on`, `jobs` y `steps`. Tambien usaremos expresiones `${{ ... }}` y contextos como `github`, `env`, `vars` y `secrets`.

```yaml
name: ci
on:
  push:
    branches: ["main"]
  pull_request:
jobs:
  checks:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: echo ${{ github.ref }}
```

## Que es un workflow y como se lee.

Un workflow es la unidad de automatizacion de GitHub Actions. Se dispara cuando ocurre uno o varios eventos definidos en `on` y organiza el trabajo que debe ejecutarse en respuesta a esos eventos.

Conviene leer un workflow de arriba hacia abajo en este orden:

- `name`: etiqueta legible de la automatizacion.
- `on`: condiciones que la activan.
- `jobs`: trabajos independientes o encadenados.
- `runs-on`: tipo de runner requerido por cada job.
- `steps`: secuencia de acciones y comandos dentro del job.

La idea importante es que un workflow no es un script lineal grande. Es una declaracion de condiciones, dependencias y pasos de ejecucion. Eso permite reusar acciones, paralelizar trabajos y aplicar reglas distintas segun rama, evento o entorno.

## Triggers, contextos y expresiones.

Los triggers responden a eventos del repositorio, por ejemplo `push`, `pull_request`, `workflow_dispatch` o `schedule`. A partir de ese evento, GitHub expone datos en distintos contextos como `github`, `runner`, `env`, `vars`, `secrets` y `matrix`.

Las expresiones `${{ ... }}` permiten usar esos datos para tomar decisiones dentro del workflow. Por ejemplo, habilitar un despliegue solo si la rama es `main`, cambiar variables por entorno o construir una matriz de versiones.

Esa combinacion de evento, contexto y expresion es lo que vuelve a GitHub Actions un DSL operativo para CI/CD y no solo un archivo YAML descriptivo.

## Ejemplo concreto: workflow para pull request y rama principal.

En un repositorio protegido, un flujo comun es este: una rama corta dispara checks con `pull_request`; despues del merge a `main`, otro evento `push` habilita pasos de promocion o despliegue controlado.

```yaml
on:
  pull_request:
    branches: ["main"]
  push:
    branches: ["main"]
```

Este patron conecta directamente el workflow tecnico con la politica del repositorio: el pull request valida; la rama principal promueve.

## Dependencias entre jobs.

Por defecto, todos los jobs de un workflow se ejecutan en paralelo. Para que un job espere a que otro termine antes de arrancar, se usa `needs:`.

```yaml
jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: ruff check .

  tests:
    runs-on: ubuntu-latest
    needs: lint          # solo corre si lint termina con exito
    steps:
      - uses: actions/checkout@v4
      - run: pytest

  deploy:
    runs-on: ubuntu-latest
    needs: [lint, tests] # espera a ambos
    steps:
      - run: echo "desplegando"
```

Esto define un grafo de dependencias: `lint` y `tests` pueden correr en paralelo; `deploy` espera a que ambos terminen con éxito. Si cualquier job del que se depende falla, los jobs descendientes se cancelan automáticamente.

`needs:` acepta un solo nombre (`needs: lint`) o una lista (`needs: [lint, tests]`). Un job sin `needs:` siempre corre en paralelo con los demás.

## Condicionales: `if:` en jobs y steps.

`if:` permite ejecutar un job o un step solo cuando se cumple una condición. Se evalúa como expresión `${{ ... }}`, aunque los corchetes son opcionales cuando no hay ambigüedad.

```yaml
jobs:
  deploy:
    runs-on: ubuntu-latest
    needs: tests
    if: github.ref == 'refs/heads/main'   # solo en main
    steps:
      - uses: actions/checkout@v4

      - name: notificar en fallo
        if: failure()                         # corre si un step anterior fallo
        run: echo "algo salio mal"

      - name: limpiar siempre
        if: always()                          # corre sin importar el resultado
        run: rm -rf /tmp/build
```

Las funciones de estado mas utiles son:

| Funcion | Cuando corre |
|---|---|
| `success()` | todos los steps anteriores pasaron (comportamiento por defecto) |
| `failure()` | al menos un step anterior fallo |
| `always()` | sin importar el resultado |
| `cancelled()` | el workflow fue cancelado |

Combinado con `needs:`, `if:` permite construir flujos completos: desplegar solo si los tests pasan y la rama es `main`, notificar solo si algo falla, limpiar recursos siempre.

## Artefactos: persistir resultados entre jobs.

Cada job corre en un runner efímero que desaparece al terminar. Si un job genera un archivo que otro necesita —un binario compilado, un reporte de cobertura, una imagen exportada— debe persistirlo como artefacto antes de que el runner se elimine.

```yaml
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: python -m build        # genera dist/
      - uses: actions/upload-artifact@v4
        with:
          name: paquete
          path: dist/

  publish:
    runs-on: ubuntu-latest
    needs: build
    steps:
      - uses: actions/download-artifact@v4
        with:
          name: paquete
          path: dist/
      - run: twine upload dist/*
```

`upload-artifact` guarda los archivos en el almacenamiento de GitHub. `download-artifact` los recupera en cualquier job posterior. Los artefactos se eliminan automáticamente a los 90 días por defecto (configurable con `retention-days:`).

Este patrón tiene una consecuencia de seguridad directa: garantiza que lo que se probó y lo que se despliega es exactamente el mismo binario, no una reconstrucción posterior.

## Matriz de versiones.

La estrategia `matrix` ejecuta el mismo job con distintos valores sin duplicar código. Es la forma idiomática de validar contra múltiples versiones de Python, sistemas operativos o configuraciones:

```yaml
jobs:
  tests:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12", "3.13"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pytest
```

GitHub Actions crea un job independiente por cada valor y los ejecuta en paralelo. Con dos dimensiones —version de Python y sistema operativo— el numero de jobs es el producto cartesiano de ambas listas.

La matriz se cubre en detalle en capitulos posteriores junto con su integración con entornos protegidos y artefactos.

## Control de ejecución: timeouts y fallos esperados.

`timeout-minutes:` limita cuánto puede correr un job o un step. Sin este límite, un proceso colgado consume tiempo de runner hasta agotar la cuota del repositorio:

```yaml
jobs:
  tests:
    runs-on: ubuntu-latest
    timeout-minutes: 15        # el job falla si no termina en 15 min
    steps:
      - uses: actions/checkout@v4
      - run: pytest
        timeout-minutes: 10    # tambien aplicable a steps individuales
```

`continue-on-error:` permite que un job o step falle sin cancelar el resto del workflow. Util para checks opcionales que deben reportar pero no bloquear:

```yaml
      - name: escaneo de seguridad (no bloqueante)
        continue-on-error: true
        run: trivy fs .
```

Usados junto con `if: failure()`, estos controles permiten construir pipelines que reportan información aunque algo falle, en lugar de cancelarlo todo ante el primer error.

In [ ]:
workflow = {
    'name': 'ci',
    'on': ['push', 'pull_request', 'workflow_dispatch'],
    'jobs': {'checks': {'runs-on': 'ubuntu-latest'}}
}

for clave, valor in workflow.items():
    print(clave, '=>', valor)

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>